In [1]:
%load_ext autoreload
%autoreload 2
%reset -f

# Imports

In [ ]:
from locallib.picarrodb import *
from locallib.query import *
from locallib.box import *
from locallib.query import *

import pandas as pd
import geopandas as gpd
import contextily as ctx
import pandas as pd
import os

import sys
import os

os.chdir('/home/sandbox/personal-repos/DA-3507/dump')
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../lib')))
from custom_pandas import *

In [ ]:
customer_name = "Cadent"
start_date = "2026-04-25"
end_date = "2026-04-26"

## Get the surveys

In [ ]:
a = get_users(customer_name, '#UserList')
b = get_surveys('#UserList',start_date = start_date, survey_table = '#TempSurvey')
c = query_SurveyH3Aggregation('#TempSurvey',table_name = '#TempBC')
b.set_child(c)
a.set_child(b)
df = a.execute(EU2_Conn, table_return = ['#TempSurvey','#TempBC'])

In [ ]:
print(len(df['#TempSurvey']))

# Rasterize the boundary

In [ ]:
# Merge SurveyArea and Breadcrumb into a single DataFrame
survey_df = df['#TempSurvey'][['SurveyId','SurveyArea']].copy()
boundary_df = df['#TempBC'][['SurveyId', 'Breadcrumb']].copy()
merged = survey_df.merge(boundary_df, on='SurveyId', how='left')
merged.BreadcrumbCounts.set_H3Resolution(10)
merged.BreadcrumbCounts.set_counts_only(True)
a = merged.BreadcrumbCounts.calculate('Breadcrumb','SurveyArea')

In [ ]:
a.to_parquet("breadcrumb_counts.parquet")

In [ ]:
# Calculate the percentage of odd 'Intersections'
summary = a.groupby('Intersections').agg({'num_cells': 'sum'}).reset_index()
summary

In [ ]:
odd_cells = summary[summary['Intersections'] % 2 == 1]['num_cells'].sum()
percentage_odd = 100 * odd_cells / total_cells if total_cells > 0 else 0
print(f"Percentage of odd intersections: {percentage_odd:.2f}%")

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely import wkt
import h3
from shapely.geometry import Polygon

# Take just the first item
first_row = merged.iloc[2]

# Convert WKT fields to geometry
surveyarea_geom = wkt.loads(first_row['SurveyArea'])
breadcrumb_geom = wkt.loads(first_row['Breadcrumb'])
survey_id = first_row['SurveyId']

# Get all H3 cells from 'a' that have the corresponding survey_id
h3_cells = a[a['SurveyId'] == survey_id]

# Generate the vcell geometries (polygon for each h3 cell, in GeoSeries)

def h3_to_geo_boundary(h3_index):
    boundary = h3.cell_to_boundary(h3_index)
    boundary = [(lat, lon) for lon, lat in boundary]
    return Polygon(boundary)

# If the h3 cell column is named something specific (e.g., 'h3index' or 'h3_cell'), update as necessary
h3_col_name = [col for col in h3_cells.columns if 'h3' in col.lower()][0]  # try to auto-find
h3_cells['geometry'] = h3_cells[h3_col_name].apply(h3_to_geo_boundary)

h3_gdf = gpd.GeoDataFrame(h3_cells, geometry='geometry', crs=4326)
print(h3_gdf)
# Create GeoDataFrames for each geometry
gdf = gpd.GeoDataFrame({'geometry': [surveyarea_geom]}, crs=4326)
breadcrumb_gdf = gpd.GeoDataFrame({'geometry': [breadcrumb_geom]}, crs=4326)

fig, ax = plt.subplots(figsize=(10, 10))

# Plot SurveyArea (polygon)
gdf.plot(ax=ax, facecolor='none', edgecolor='blue', linewidth=2, label='Survey Area')

# Plot Breadcrumb (line)
breadcrumb_gdf.plot(ax=ax, color='red', linewidth=1, label='Breadcrumb')

# Discretize the color mapping for integer "Counts" using a ListedColormap with boundaries for each count value.
import numpy as np
import matplotlib as mpl

if 'Counts' in h3_gdf.columns:
    counts = h3_gdf['Counts'].fillna(0).astype(int)
    unique_counts = np.sort(h3_gdf['Counts'].dropna().unique())
    # Set boundaries as half-step between each integer count for clear binning; pad at ends for color limits.
    if len(unique_counts) > 1:
        boundaries = np.concatenate((
            [unique_counts[0] - 0.5], 
            (unique_counts[:-1] + unique_counts[1:]) / 2, 
            [unique_counts[-1] + 0.5]
        ))
    else:
        boundaries = [unique_counts[0] - 0.5, unique_counts[0] + 0.5]
    cmap = mpl.cm.get_cmap('YlOrRd', len(unique_counts))
    norm = mpl.colors.BoundaryNorm(boundaries, cmap.N)
    h3_gdf.plot(
        ax=ax,
        column='Counts',
        cmap=cmap,
        alpha=0.6,
        edgecolor='green',
        linewidth=0.5,
        legend=True,
        norm=norm,
        label='H3 Cells'
    )
else:
    h3_gdf.plot(
        ax=ax,
        alpha=0.6,
        edgecolor='green',
        linewidth=0.5,
        legend=True,
        label='H3 Cells'
    )

plt.legend(['Survey Area', 'Breadcrumb'])
plt.title("Survey Area (blue) and Breadcrumb (red) - First Item Only")
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()